# Delta Demo — Episode 11: Duplicate Handling (Self-Contained)
### "Why Did Removing ONE Duplicate Rewrite the ENTIRE Table?"

**This notebook needs nothing else run first.** It creates its own table, at its own path, from scratch, in three versions: a fresh load, a schema change, and a duplicate-detection-and-fix cycle. Nothing here depends on any other notebook's state.

**Learning Outcome:** By the end of this episode, viewers should be able to detect duplicate rows in a Delta table using real evidence, and understand why `CREATE OR REPLACE TABLE ... AS SELECT DISTINCT` is a full logical rewrite — not an incremental fix.

**Core Question:** A bad upstream feed just sent us the same record twice. Can we trust our reports — and how do we clean this up without breaking anything else?

**How this notebook is organized:** cells marked **VERIFY** check a specific claim programmatically and print a pass/fail result — nothing here rests on visual inspection alone.

---
**This notebook is self-contained.** It creates and uses its own Delta table (`employees_ep11`) in a dedicated path. You can run it independently without executing any previous episodes. Re-running the notebook safely resets only its own demo data — nothing outside this path is ever touched.

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo table:** `employees_ep11`

**Deletes only its own demo data:** Yes

---

### Today's Journey
✔ Create a clean Delta table

↓

✔ Simulate a duplicate

↓

✔ Detect the problem

↓

✔ Remove duplicates

↓

✔ Inspect Parquet files

↓

✔ Read the Delta Log

↓

✔ Understand why the table rewrote

## Step 0 — Setup (Self-Contained Reset)
This deletes and recreates a dedicated path just for this episode — `employees_ep11` — so this notebook is completely independent of any other notebook's table state. Safe to re-run from scratch any time.

In [0]:
%sh
# Clear any leftover copy from a previous run of THIS notebook, so we start
# from a genuinely clean slate every time.
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep11

In [0]:
%sql
-- Safe to re-run: creates the schema and Volume only if they don't already exist.
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

# =====================================================
# STEP 1 — Version 1: Create Baseline (Fresh Table, 5 Records)
# =====================================================
Sample data — five employees, no history, no prior operations.

In [0]:
%python
from pyspark.sql import functions as F

table_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep11"

# Sample data: eno, ename, sal
v1 = spark.createDataFrame(
    [
        (1, 'Ravi', 25000),
        (2, 'Sridevi', 23000),
        (3, 'Uma', 35000),
        (4, 'Srik', 32000),
        (5, 'Kanth', 28000),
    ],
    "eno INT, ename STRING, sal INT"
).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

v1.write.format("delta").mode("overwrite").save(table_path)

### Verify Version 1

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` ORDER BY eno;

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11`;

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep11/_delta_log/
echo ""
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep11/*.parquet

# =====================================================
# STEP 2 — Version 2: Schema Evolution (Add Column)
# =====================================================
HR wants an Age column. Let's give it to them, and prove it's a metadata-only change — no data files touched.

In [0]:
%python
schema_before = spark.read.format("delta").load(table_path).schema
import glob
files_before_v2 = sorted(f.split("/")[-1] for f in glob.glob(f"{table_path}/*.parquet"))
print("Schema BEFORE:", schema_before)
print("Files BEFORE:", files_before_v2)

In [0]:
%sql
ALTER TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` ADD COLUMNS (age INT);

### Query Immediately — No Backfill Has Run

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` ORDER BY eno;

### VERIFY — Metadata-Only Change

In [0]:
%python
import json

schema_after = spark.read.format("delta").load(table_path).schema
files_after_v2 = sorted(f.split("/")[-1] for f in glob.glob(f"{table_path}/*.parquet"))

log_files = sorted(glob.glob(f"{table_path}/_delta_log/*.json"))
with open(log_files[-1]) as f:
    action_types = [list(json.loads(line).keys())[0] for line in f]

schema_changed = "age" in schema_after.fieldNames() and "age" not in schema_before.fieldNames()
files_unchanged = files_before_v2 == files_after_v2
metadata_only = "metaData" in action_types and "add" not in action_types and "remove" not in action_types

print(f"Schema changed correctly: {schema_changed}")
print(f"Files unchanged: {files_unchanged}")
print(f"Commit is metadata-only: {metadata_only}")

if schema_changed and files_unchanged and metadata_only:
    print("\n✅ VERIFIED: schema changed, zero data files touched.")
else:
    print("\n❌ NOT VERIFIED — investigate above.")

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11`;

# =====================================================
# STEP 3 — Version 3: Simulate the Duplicate
# =====================================================
**Keep this sentence in mind for everything that follows:** *Delta isn't rewriting because of the duplicate. Delta is rewriting because you asked it to replace the table.*

### Step 3 — Simulate the Bad Upstream Feed

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` VALUES (5, 'Kanth', 28000, NULL);

### The Business Symptom

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` ORDER BY eno;

### Step 4 — Detect the Duplicate

In [0]:
%sql
SELECT eno, COUNT(*) AS row_count
FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11`
GROUP BY eno
HAVING COUNT(*) > 1;

### VERIFY — Exactly One Duplicated `eno`

In [0]:
%python
dupes = (
    spark.read.format("delta").load(table_path)
    .groupBy("eno").count()
    .filter(F.col("count") > 1)
    .collect()
)
if len(dupes) == 1 and dupes[0]["eno"] == 5:
    print(f"✅ VERIFIED: exactly one duplicated eno — eno=5, appearing {dupes[0]['count']} times.")
else:
    print("❌ NOT VERIFIED:", dupes)

### What We Expect to Happen (Picture This Before Running It)
```
BEFORE CTAS                         AFTER CTAS

  File A ┐                            File A  (inactive, still on disk)
  File B ├─ active, contains          File B  (inactive, still on disk)
  File C ┘  the duplicate row         File C  (inactive, still on disk)
                                            │
                                            ▼
                                      File D  ← NEW, active,
                                                deduped, 5 rows
```
Files don't disappear — they become inactive, not deleted. They're still physically on disk. That's exactly what sets up the *next* episode.

In [0]:
%python
files_before_dedupe = sorted(f.split("/")[-1] for f in glob.glob(f"{table_path}/*.parquet"))
print(f"Physical Parquet files before dedupe: {len(files_before_dedupe)}")

### Step 5 — Fix (CTAS Dedupe)
`SELECT DISTINCT` collapses the two eno=5 rows into one. `CREATE OR REPLACE TABLE ... AS SELECT` recomputes the entire table from that query's result — NOT a targeted DELETE of one row.

In [0]:
%sql
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` AS
SELECT DISTINCT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11`;

### Did Delta Record a New Version?
Check immediately — before looking at anything else.

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11`;

**Yes — a new version was recorded**, same as every other operation. Now let's see exactly what happened to the files, then read what got written into the log.

### Verify the Result

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep11` ORDER BY eno;

### VERIFY — Duplicate Is Gone

In [0]:
%python
row_count_final = spark.read.format("delta").load(table_path).count()
dupes_final = (
    spark.read.format("delta").load(table_path)
    .groupBy("eno").count().filter(F.col("count") > 1).collect()
)
if row_count_final == 5 and len(dupes_final) == 0:
    print(f"✅ VERIFIED: {row_count_final} rows, no duplicated eno remains.")
else:
    print(f"❌ NOT VERIFIED — row_count={row_count_final}, dupes={dupes_final}")

### Step 6 — Inspect Storage: Was This a Full Rewrite?
Reads `operationMetrics` from `DESCRIBE HISTORY` for the exact commit — NOT a raw physical directory scan, which would be misleading (orphaned files from earlier compactions can still be sitting on disk and would show false overlap).

In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
latest = history_df.orderBy(history_df.version.desc()).first()
metrics = dict(latest['operationMetrics'])

num_removed = int(metrics.get('numRemovedFiles', 0))
# CTAS operations report the active file count under 'numFiles', not
# 'numAddedFiles' — that key only exists for row-level DML (UPDATE/DELETE/MERGE).
# Check both so this cell works correctly regardless of operation type.
num_added = int(metrics.get('numAddedFiles', metrics.get('numFiles', 0)))

print(f"Operation: {latest['operation']}")
print(f"Files removed from ACTIVE snapshot: {num_removed}")
print(f"Files added to ACTIVE snapshot: {num_added}")

if num_removed > 0 and num_added > 0:
    print("\n✅ VERIFIED: full logical rewrite — every previously active file")
    print("   was removed from the snapshot, a fresh file set was added.")
else:
    print("\n❌ NOT VERIFIED — check operationMetrics above.")

### Why Did Removing ONE Duplicate Rewrite the ENTIRE Table?
**Short answer: it didn't rewrite because of the duplicate. It rewrote because `CREATE OR REPLACE TABLE AS SELECT` tells Delta to build a brand-new table from the query result.**

**`UPDATE`/`DELETE` are row-level DML** — Delta knows exactly which row changed, so it can surgically attach a Deletion Vector to just one file.

**`CREATE OR REPLACE TABLE ... AS SELECT` is a CTAS operation, not row-level DML.** For this operation, Delta doesn't attempt to compare the new query result with the existing table row by row. Instead, it treats the query output as the new table contents and writes a new set of data files.

**The mental model to remember:**

| Operation | Delta's Knowledge | Result |
|---|---|---|
| `UPDATE eno=5` | "I know exactly which row changed." | Targeted change (Deletion Vector) |
| `DELETE WHERE eno=5` | "I know exactly which row changed." | Targeted change (Deletion Vector) |
| `CREATE OR REPLACE TABLE AS SELECT DISTINCT` | "Here's a brand-new table definition." | Full rewrite |

**This episode isn't claiming duplicate removal always rewrites the table.** A targeted `DELETE` on the duplicate's specific row, or a dedup-aware `MERGE`, would NOT have triggered a full rewrite. This is about what THIS specific strategy does — because of what kind of operation it is, not what it happens to accomplish.

### Step 7 — Inspect the Delta Log (JSON)
If Delta rebuilt the entire table... what exactly did it record in the transaction log?

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep11/_delta_log/*.json

In [0]:
%python
log_files = sorted(glob.glob(f"{table_path}/_delta_log/*.json"))
latest_commit_path = log_files[-1]
print(f"Reading: {latest_commit_path.split('/')[-1]}\n")

action_counts = {}
with open(latest_commit_path) as f:
    for line in f:
        action = json.loads(line)
        t = list(action.keys())[0]
        action_counts[t] = action_counts.get(t, 0) + 1

print("Action counts in this commit:")
for t, c in action_counts.items():
    print(f"  {t}: {c}")

### VERIFY — History Is Still Intact
CTAS is a new commit, not a DROP + CREATE.

In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
min_version = history_df.agg(F.min("version")).collect()[0][0]
total_versions = history_df.count()
if min_version == 0:
    print(f"✅ VERIFIED: full history intact — {total_versions} versions, starting from version 0.")
else:
    print("❌ Unexpected — investigate.")

### The Closing Reveal — Active vs. Physical
How many files does a query actually read right now, versus how many are genuinely sitting on disk?

In [0]:
%python
# CTAS commits use 'numFiles' for the active file count, not 'numAddedFiles'
# (that key only applies to row-level DML operations like UPDATE/DELETE/MERGE).
# Confirmed here via two independent methods agreeing on the same number.
active_files = int(dict(latest['operationMetrics']).get('numFiles', 0))
total_physical_files = len(glob.glob(f"{table_path}/*.parquet"))

print(f"Files ACTUALLY being read by a query right now: {active_files}")
print(f"Files PHYSICALLY sitting on disk right now: {total_physical_files}")
print(f"\nOrphaned garbage from everything we've done in this notebook: "
      f"{total_physical_files - active_files} files")

# =====================================================
# STEP 8 — Enterprise Reality
# =====================================================
> "Duplicate rows usually come from retries, late-arriving files, or upstream systems replaying data — not from a mistake in your own pipeline. How you fix them matters: Delta rewrites because you replaced the table's definition — not because it had to."

**We proved that CTAS rewrote the table, created a new Delta version, and left the old Parquet files sitting on disk.**

If those old files still exist... can we actually recover yesterday's table?

That's exactly what we'll explore in the next episode: Time Travel and RESTORE.